<a href="https://colab.research.google.com/github/aleumprerna/AgentToAgent/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Government ID OCR — DeepSeek-OCR
Run cells in order. Use only IDs you are authorized to process. Images and results remain local; the model downloads from Hugging Face.

## 1. Install dependencies
Use a CUDA-enabled NVIDIA GPU. Install PyTorch compatible with your CUDA driver first if necessary.

In [1]:
%pip uninstall -y transformers tokenizers torch torchvision
%pip install --no-cache-dir --force-reinstall \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "accelerate" \
    "einops" \
    "addict" \
    "easydict" \
    "sentencepiece" \
    "torch" \
    "torchvision"

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: tokenizers 0.20.3
Uninstalling tokenizers-0.20.3:
  Successfully uninstalled tokenizers-0.20.3
Found existing installation: torch 2.14.1
Uninstalling torch-2.14.1:
  Successfully uninstalled torch-2.14.1
Found existing installation: torchvision 0.29.1
Uninstalling torchvision-0.29.1:
  Successfully uninstalled torchvision-0.29.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 254.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 289.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 254.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 181.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 323.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Configure paths

In [2]:
from pathlib import Path
import torch, json, re
from PIL import Image

IMAGE_PATH = Path("/content/adhar.webp")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "deepseek-ai/DeepSeek-OCR"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH.resolve()}"


KeyboardInterrupt: 

## 3. Preview the ID image

In [ ]:
from IPython.display import display
image = Image.open(IMAGE_PATH).convert("RGB")
print("Image dimensions:", image.size)
display(image)


In [ ]:
pip install torch

## 4. Load DeepSeek-OCR
This model uses custom model code from its Hugging Face repository. Review and trust that repository before running. For safety, do not use arbitrary untrusted model repositories.

In [ ]:
import torchvision
from transformers import AutoModel, AutoTokenizer
import torch

if not torch.cuda.is_available():
    raise RuntimeError("DeepSeek-OCR inference in this notebook requires an NVIDIA CUDA GPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, use_safetensors=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    _attn_implementation="eager",
).eval().cuda()

## 5. Extract all visible text
DeepSeek-OCR uses its custom `infer` method and writes output files to the chosen directory.

In [ ]:
PROMPT = "<image>\nFree OCR."
result = model.infer(
    tokenizer,
    prompt=PROMPT,
    image_file=str(IMAGE_PATH.resolve()),
    output_path=str(OUTPUT_DIR.resolve()),
    base_size=1024,
    image_size=640,
    crop_mode=True,
    save_results=True,
    test_compress=False,
)
print("OCR response:")
print(result)


## 6. Save OCR output as TXT and JSON

In [ ]:
ocr_text = result if isinstance(result, str) else ""
if not ocr_text.strip():
    candidate_files = sorted(OUTPUT_DIR.glob("*.md"), key=lambda x: x.stat().st_mtime, reverse=True)
    if candidate_files:
        ocr_text = candidate_files[0].read_text(encoding="utf-8")
if not ocr_text.strip():
    print("Check files in output/ for the OCR result; this model version did not return text directly.")
else:
    (OUTPUT_DIR / "ocr_text.txt").write_text(ocr_text, encoding="utf-8")
    payload = {"source_image": IMAGE_PATH.name, "model": MODEL_ID, "extracted_text": ocr_text}
    (OUTPUT_DIR / "ocr_result.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(ocr_text)
    print("Saved TXT and JSON in", OUTPUT_DIR.resolve())
